[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/linear_regression_exercise_mpg.ipynb)

# Linear regression: predicting fuel consumption

In this exercise you will build a multiple linear regression model to predict a car's fuel efficiency, measured in **miles per gallon (mpg)**, using the [Auto MPG dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg) from the UCI Machine Learning Repository.

Compared with the wine quality exercise, this dataset has a few real-world problems you have to deal with:

* **missing values**, encoded as `?` instead of an empty field;
* a **categorical** variable (`origin`) that is stored as a number;
* a text column that should not go into the model;
* relationships that are **not perfectly linear**.

### How this notebook works

* Cells with `...` or `# TODO` are for you to complete.
* Cells marked **Predict first** ask you to write what you *expect* **before** running the code. You should rely on your reasoning, not on whether the guess was right.
* Cells marked **Explain** must be answered in your own words and must refer to the **numbers you obtained**. Every student gets a slightly different subset of the data, so your numbers will be different from your colleagues'.
* `check(...)` cells test your work. If one fails, fix the cells above it before moving on.

## 0. Your student number

Write your student number below. It is used as the random seed, so it decides which cars end up in your dataset and in your test set. **Do not change it after you start.**

In [30]:
STUDENT_ID = 71113  # TODO: replace with your student number, e.g. 12345

assert isinstance(STUDENT_ID, int), "Set STUDENT_ID to your student number (an integer)."
SEED = STUDENT_ID % (2**32)

In [31]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

## 1. Dataset

### 1.1 Download

The data is a plain text file where the columns are separated by **whitespace** (not commas) and there is **no header line**. The column names come from the [dataset description](https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.names).

In [32]:
import os
import urllib.request

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
FILE = "auto-mpg.data"
if not os.path.exists(FILE):
    urllib.request.urlretrieve(URL, FILE)

column_names = ["mpg", "cylinders", "displacement", "horsepower", "weight",
                "acceleration", "model_year", "origin", "car_name"]

### 1.2 Load

Open `auto-mpg.data` in a text editor (or run `!head auto-mpg.data`) and have a look at the raw file. Then complete the [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) call:

* the separator is one or more spaces (hint: the regular expression `r"\s+"`);
* the file has no header, so you have to pass the column names;
* missing values are written as `?`. Use the `na_values` parameter so pandas turns them into `NaN`.

In [33]:
raw = pd.read_csv(FILE,
                  sep=r"\s+",
                  names=column_names,
                  na_values="?")

# Each student works on a different random 90% of the cars
df = raw.sample(frac=0.9, random_state=SEED).reset_index(drop=True)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,22.0,6,198.0,95.0,2833.0,15.5,70,1,plymouth duster
1,31.3,4,120.0,75.0,2542.0,17.5,80,3,mazda 626
2,13.0,8,350.0,145.0,3988.0,13.0,73,1,chevrolet malibu
3,33.5,4,151.0,90.0,2556.0,13.2,79,1,pontiac phoenix
4,34.2,4,105.0,70.0,2200.0,13.2,79,1,plymouth horizon


In [34]:
check(raw.shape == (398, 9), "raw dataset has 398 rows and 9 columns")
check(raw["horsepower"].dtype == float, "horsepower was read as a number (the '?' became NaN)")

✅ raw dataset has 398 rows and 9 columns
✅ horsepower was read as a number (the '?' became NaN)


### 1.3 Understanding the dataset

Use `shape`, `info()` and `describe()` to explore the data.

In [35]:
# shape, info and descriptive statistics
raw.shape

(398, 9)

In [36]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    float64
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car_name      398 non-null    object 
dtypes: float64(5), int64(3), object(1)
memory usage: 28.1+ KB


In [37]:
raw.describe()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398.000000,398.000000,398.000000,392.000000,398.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.469388,2970.424623,15.568090,76.010050,1.572864
std,7.815984,1.701004,104.269838,38.491160,846.841774,2.757689,3.697627,0.802055
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,1.000000
25%,17.500000,4.000000,104.250000,75.000000,2223.750000,13.825000,73.000000,1.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000,1.000000
75%,29.000000,8.000000,262.000000,126.000000,3608.000000,17.175000,79.000000,2.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000,3.000000


**Explain:** What are the units of `weight` and `displacement`? What values does `origin` take, and what do you think they mean? Look at the [dataset description](https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.names) if you need to.

**Your answer:**

- `weight` (peso): É medido em libras (lbs - pounds). No padrão americano, é comum expressar o peso de veículos nessa unidade (por exemplo, 3504 lbs equivalem a aproximadamente 1589 kg).
- `displacement` (deslocamento/cilindrada do motor): É medido em polegadas cúbicas (cu in - cubic inches). É uma unidade de volume usada para medir o volume total dos cilindros do motor (por exemplo, 307 polegadas cúbicas equivalem a aproximadamente 5.0 litros).



### 1.4 Missing values

Count the missing values in each column.

In [38]:
# TODO: number of missing values per column
raw.isnull().sum()

,0
mpg,0
cylinders,0
displacement,0
horsepower,6
weight,0
acceleration,0
model_year,0
origin,0
car_name,0


There are two common ways to deal with the missing values:

1. **drop** the rows that have them ([`dropna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html));
2. **impute** them, for example with the median of the column ([`fillna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)).

Pick one, apply it, and justify your choice below.

In [39]:
# TODO: handle the missing values of df
df = raw.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        392 non-null    float64
 5   acceleration  392 non-null    float64
 6   model_year    392 non-null    int64  
 7   origin        392 non-null    int64  
 8   car_name      392 non-null    object 
dtypes: float64(5), int64(3), object(1)
memory usage: 30.6+ KB


In [40]:
check(df.isnull().sum().sum() == 0, "no missing values left")

✅ no missing values left


**Explain:** Which strategy did you choose? How many rows did you have before and after? When would the other strategy be the better choice?

**Your answer:**

I used dropna strategy, because is more simple and i knew how to use. Before i had 398 rows and now i have 392 rows. The other strategy is better when you have a small dataset and drop the rows will be affect the results.


### 1.5 Preparing the columns

`car_name` is a text column that is (almost) unique for every car. Remove it from `df`.

In [41]:
# TODO: drop the car_name column
df = df.drop(columns=["car_name"])
df.shape

(392, 8)

`origin` is stored as 1, 2 and 3, but it is a **category**, not a quantity.

**Predict first:** If we left `origin` as the number 1, 2 or 3, what would the model be assuming about the difference between American (1), European (2) and Japanese (3) cars? Write your answer *before* doing the next step.

**Your answer:**

if we left `origin` as numbers, the model will see this values as a continuous numerical scale and not a categorical attribute


Replace the numbers with the region names (`1 → "usa"`, `2 → "europe"`, `3 → "japan"`) using [`map`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html), then turn the column into **one-hot** (dummy) variables with [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html). Use `drop_first=True` and `dtype=float`.

In [ ]:
df["origin"] = df["origin"].map(...)  # TODO
df = pd.get_dummies(...)               # TODO
df.head()

In [ ]:
check("origin" not in df.columns and "car_name" not in df.columns, "origin and car_name are no longer columns")
check(sum(c.startswith("origin_") for c in df.columns) == 2, "origin became 2 dummy columns")

**Explain:** Why do we use `drop_first=True`? Which region was dropped, and what does that region become for the model?

**Your answer:**



### 1.6 Visual exploration

Draw a [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html) of `mpg`, `displacement`, `horsepower`, `weight` and `acceleration`, and a [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) of the correlation matrix of all the columns (use `annot=True`).

In [ ]:
# TODO: pairplot

In [ ]:
# TODO: correlation heatmap

**Explain:**
1. Which variable has the strongest correlation with `mpg`? Is it positive or negative, and does that make physical sense?
2. Look at `mpg` vs `weight` and `mpg` vs `horsepower` in the pairplot. Is the relationship a straight line?
3. Find two input variables that are strongly correlated **with each other**. Why could that be a problem for interpreting the coefficients of a linear model?

**Your answer:**



## 2. Simple linear regression: mpg vs weight

Start with a single variable, as in `linear_regression.ipynb`. Use `weight` as $x$ and `mpg` as $y$.

**Predict first:** Look at the scatter plot. Roughly how many mpg do you expect a car to lose for every extra 1000 lbs? Write your guess, then fit the model.

**Your answer:**



In [ ]:
from sklearn.linear_model import LinearRegression

x_w = df[["weight"]]   # a DataFrame with one column, shape (n_samples, 1)
y = df["mpg"]

simple = LinearRegression()
# TODO: fit the model

print("slope:    ", ...)  # TODO
print("intercept:", ...)  # TODO

Draw the data together with the fitted line.

In [ ]:
wfit = np.linspace(df["weight"].min(), df["weight"].max(), 100)
# TODO: predict mpg for wfit, then scatter the data and plot the line

**Explain:** Write the slope as "a car that is 1000 lbs heavier gets ___ mpg less". Was your guess close? Does the line fit well for the lightest and the heaviest cars?

**Your answer:**



## 3. Multiple linear regression

### 3.1 Train/test split

Separate the target (`mpg`) from the input variables and split the data into 70% training and 30% test. Use `random_state=SEED`.

In [ ]:
from sklearn.model_selection import train_test_split

X = ...  # TODO: every column except mpg
y = ...  # TODO
X_train, X_test, y_train, y_test = ...  # TODO

In [ ]:
check("mpg" not in X.columns, "the target is not among the inputs")
check(abs(len(X_test) / len(X) - 0.3) < 0.01, "test set is 30% of the data")

### 3.2 Training

Train a `LinearRegression` model on the training set and show its coefficients in a table (a `pd.Series` indexed by the column names works well), sorted by value.

In [ ]:
lm = LinearRegression()
# TODO: fit on the training set

coefs = ...  # TODO
coefs

**Explain:**
1. What is the coefficient of `model_year` and what does it mean in plain words?
2. Compare the coefficient of `weight` here with the slope of the simple model in Section 2. Why are they different?
3. Can you say that the variable with the largest coefficient is the most important one? (Hint: look at the units of each variable.)

**Your answer:**



### 3.3 Evaluation

Before looking at the model's error we need something to compare it with. A **baseline** model always predicts the *mean mpg of the training set*, whatever the car.

**Predict first:** What $R^2$ do you expect the baseline to get? What $R^2$ do you expect the linear model to get?

**Your answer:**



Complete the function so that it returns the MAE, RMSE and $R^2$ of a set of predictions (use [`sklearn.metrics`](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)). Then evaluate the baseline and the linear model on the **test** set.

In [ ]:
from sklearn import metrics

def evaluate(y_true, y_pred):
    mae = ...   # TODO
    rmse = ...  # TODO
    r2 = ...    # TODO
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

baseline_pred = ...  # TODO: an array with len(y_test) copies of the training mean
lm_pred = ...        # TODO

results = pd.DataFrame({
    "baseline": evaluate(y_test, baseline_pred),
    "linear": evaluate(y_test, lm_pred),
}).T
results

In [ ]:
check(abs(results.loc["baseline", "R2"]) < 0.1, "baseline R2 is close to 0")
check(results.loc["linear", "RMSE"] < results.loc["baseline", "RMSE"], "linear model beats the baseline")

**Explain:** Why is the baseline $R^2$ close to zero (and possibly a little negative)? Using your RMSE, how far off (in mpg) is a typical prediction of the linear model?

**Your answer:**



### 3.4 Residuals

A residual is $y - \hat{y}$. Plot the residuals of the test set against the predicted values, with a horizontal line at 0.

In [ ]:
residuals = ...  # TODO

# TODO: scatter plot of lm_pred (x axis) vs residuals (y axis) and a horizontal line at 0

**Explain:** If the model were a good description of the data, the residuals would look like random noise around 0. Do you see a pattern (for example a "U" shape, or a spread that grows)? What does it tell you about the model?

**Your answer:**



## 4. Improve the model

The plots in Sections 1.6 and 3.4 suggest that the relationship is not linear. Try **at least two** ideas to lower the test RMSE and record each result in the `experiments` table. Some ideas:

* add squared terms, e.g. `weight**2` or `horsepower**2`;
* predict `np.log(mpg)` instead of `mpg` (remember to convert the predictions back with `np.exp` before computing the metrics!);
* predict fuel consumption `1 / mpg` instead of `mpg` (again, convert back);
* remove variables that are redundant.

Always fit on the training set and evaluate on the test set, using the same split as before.

In [ ]:
experiments = {"linear (all features)": evaluate(y_test, lm_pred)}

# Idea 1
# TODO

# Idea 2
# TODO

pd.DataFrame(experiments).T.sort_values("RMSE")

**Explain:** Which idea worked best on *your* test set? Why do you think it helped? Is the model with squared terms still a "linear" regression?

**Your answer:**



## 5. Summary

Run the cell below and do not edit it. Your teacher can reproduce these numbers from your student number.

In [ ]:
print("Student:", STUDENT_ID)
print("Rows after cleaning:", len(df))
print(pd.DataFrame(experiments).T.round(3).to_string())

### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**

